# Give me the base ᯓ★🎸: <br> Technology Classification As Baseline

## Overview

**Purpose:** This notebook establishes a baseline model to predict CO₂ emission intensity using production technology as the sole predictor.

**Research Question:** How much of emission variance can technology alone explain?

**Model Type:** Decision tree with binary classification:
- **EAF** (Electric Arc Furnace) - cleaner technology
- **BF-BOF** (Blast Furnace-Basic Oxygen Furnace) - traditional technology

**Statistical Validation:** We use an independent samples t-test to verify that emission differences between technology groups are statistically significant (not due to chance).

**Why This Matters:** 
- Establishes a basis for more complex models
- Quantifies the fundamental role of technology in emissions
- Provides context for interpreting external drivers (carbon pricing, policy) and internal factors (corporate commitments)

## Methodology: Per-Company Consistent Scope 2 Approach

**Challenge:** Companies report Scope 2 emissions using different methods:
- **Location-based**: Actual grid emissions (better for cross-company comparison)
- **Market-based**: After renewable energy purchases (reflects company choices)

**Solution:** Each company uses ONE consistent method throughout their time series:
1. IF company has complete location-based data → Use location
2. IF NOT, but has complete market-based data → Use market
3. This maintains temporal consistency WITHIN companies while maximizing data availability ACROSS companies

**Result:**
- ✓ No mixing methods within companies (preserves trends)
- ✓ Maximum data utilization across companies
- ✓ Most companies use location-based (enables comparison)

## 1. Setup & Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_style('whitegrid')

# Load emissions data
data_all = pd.read_csv('EDA/emissions_data.txt', sep='\t', encoding='latin-1')

print(f"📂 Data loaded: {len(data_all)} rows, {len(data_all.columns)} columns")
print(f"🏭 Companies: {data_all['company'].nunique()}")
print(f"📅 Years: {data_all['year'].min():.0f}-{data_all['year'].max():.0f}")

📂 Data loaded: 129 rows, 24 columns
🏭 Companies: 15
📅 Years: 2013-2024


## 2. Determine Best Methodology Per Company

Select location-based or market-based Scope 2 for each company based on data completeness.

In [2]:
# Analyze data availability per company
company_methods = []

for company in data_all['company'].dropna().unique():
    company_data = data_all[data_all['company'] == company]
    
    # Count available years for each method
    n_years = len(company_data)
    n_location = company_data['intensity_location_co2e'].notna().sum()
    n_market = company_data['intensity_market_co2e'].notna().sum()
    
    # Select method with most complete data
    use_method = 'location' if n_location >= n_market else 'market'
    n_usable = max(n_location, n_market)
    
    company_methods.append({
        'company': company,
        'total_years': n_years,
        'location_available': n_location,
        'market_available': n_market,
        'use_method': use_method,
        'usable_obs': n_usable,
        'completeness': f"{n_usable}/{n_years}"
    })

method_df = pd.DataFrame(company_methods).sort_values('usable_obs', ascending=False)

print("\n📊 Methodology Selection per Company:")
print(method_df.to_string(index=False))
print(f"\n✅ Location-based: {(method_df['use_method']=='location').sum()} companies")
print(f"✅ Market-based: {(method_df['use_method']=='market').sum()} companies")


📊 Methodology Selection per Company:
                    company  total_years  location_available  market_available use_method  usable_obs completeness
              ArcelorMittal           12                  12                 0   location          12        12/12
                  Outokumpu           12                   3                12     market          12        12/12
                       SSAB           12                  12                 0   location          12        12/12
              Salzgitter AG           12                   9                 2   location           9         9/12
       ArcelorMittal Europe            7                   7                 0   location           7          7/7
                Voestalpine            7                   7                 6   location           7          7/7
       Tata Steel Nederland            6                   6                 0   location           6          6/6
                Celsa Group            5  

## 3. Build Clean Dataset

Apply per-company methodology and prepare data for analysis.

In [3]:
# Create method mapping
method_map = dict(zip(method_df['company'], method_df['use_method']))

# Build clean dataset
data_clean_list = []

for company, method in method_map.items():
    company_data = data_all[data_all['company'] == company].copy()
    
    # Select intensity column based on method
    intensity_col = f'intensity_{method}_co2e'
    
    # Keep only rows with valid intensity data
    valid_data = company_data[company_data[intensity_col].notna()].copy()
    valid_data['intensity'] = valid_data[intensity_col]
    valid_data['method_used'] = method
    
    data_clean_list.append(valid_data)

data_clean = pd.concat(data_clean_list, ignore_index=True)

# Simplify technology classification to binary
data_clean['tech_binary'] = data_clean['technology'].apply(
    lambda x: 'EAF' if 'EAF' in str(x) else 'BF-BOF'
)

print(f"\n✅ Clean dataset: {len(data_clean)} observations")
print(f"   Companies: {data_clean['company'].nunique()}")
print(f"   Years: {data_clean['year'].min():.0f}-{data_clean['year'].max():.0f}")
print(f"\n🔧 Technology distribution:")
print(data_clean['tech_binary'].value_counts())


✅ Clean dataset: 84 observations
   Companies: 11
   Years: 2013-2024

🔧 Technology distribution:
tech_binary
BF-BOF    62
EAF       22
Name: count, dtype: int64


## 4. Statistical Analysis

### 4.1 Descriptive Statistics

In [4]:
# Calculate statistics by technology
tech_stats = data_clean.groupby('tech_binary')['intensity'].agg([
    ('N', 'count'),
    ('Mean', 'mean'),
    ('Std', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).round(3)

print("📊 Descriptive Statistics by Technology:\n")
print(tech_stats.to_string())

# Calculate key metrics
eaf_mean = data_clean[data_clean['tech_binary'] == 'EAF']['intensity'].mean()
bof_mean = data_clean[data_clean['tech_binary'] == 'BF-BOF']['intensity'].mean()
difference = bof_mean - eaf_mean
pct_lower = (difference / bof_mean) * 100

print(f"\n🔑 Key Findings:")
print(f"   EAF average:    {eaf_mean:.3f} tCO₂e/t")
print(f"   BF-BOF average: {bof_mean:.3f} tCO₂e/t")
print(f"   Difference:     {difference:.3f} tCO₂e/t ({pct_lower:.1f}% lower for EAF)")

📊 Descriptive Statistics by Technology:

              N   Mean    Std   Min    Max
tech_binary                               
BF-BOF       62  1.805  0.271  1.45  2.587
EAF          22  0.597  0.282  0.20  0.990

🔑 Key Findings:
   EAF average:    0.597 tCO₂e/t
   BF-BOF average: 1.805 tCO₂e/t
   Difference:     1.208 tCO₂e/t (66.9% lower for EAF)


### 4.2 Statistical Significance Test (t-test)

**Null Hypothesis (H₀):** No difference in emissions between EAF and BF-BOF  
**Alternative Hypothesis (H₁):** EAF has significantly lower emissions than BF-BOF

We use an independent samples t-test to determine if observed differences are statistically significant.

In [5]:
# Separate data by technology
eaf_data = data_clean[data_clean['tech_binary'] == 'EAF']['intensity']
bof_data = data_clean[data_clean['tech_binary'] == 'BF-BOF']['intensity']

# Perform t-test
t_stat, p_value = stats.ttest_ind(eaf_data, bof_data)

print(f"📊 Independent Samples T-Test:")
print(f"   t-statistic: {t_stat:.4f}")
print(f"   p-value:     {p_value:.6f}")

# Interpret significance
if p_value < 0.001:
    sig_level = "Highly significant (p < 0.001) ***"
elif p_value < 0.01:
    sig_level = "Significant (p < 0.01) **"
elif p_value < 0.05:
    sig_level = "Significant (p < 0.05) *"
else:
    sig_level = "Not significant (p ≥ 0.05)"

print(f"   Result:      {sig_level}")
print(f"\n✅ Conclusion: Technology differences are {sig_level.lower()}")

📊 Independent Samples T-Test:
   t-statistic: -17.7834
   p-value:     0.000000
   Result:      Highly significant (p < 0.001) ***

✅ Conclusion: Technology differences are highly significant (p < 0.001) ***


### 4.3 Model Performance (R²)

Calculate how much variance in emissions the baseline model explains.

In [6]:
# Create predictions using group means
data_clean['predicted'] = data_clean['tech_binary'].map({
    'EAF': eaf_mean,
    'BF-BOF': bof_mean
})
data_clean['residual'] = data_clean['intensity'] - data_clean['predicted']

# Calculate R²
y_mean = data_clean['intensity'].mean()
SS_total = ((data_clean['intensity'] - y_mean) ** 2).sum()
SS_residual = ((data_clean['residual']) ** 2).sum()
r_squared = 1 - (SS_residual / SS_total)

# Calculate error metrics
mae = np.abs(data_clean['residual']).mean()
rmse = np.sqrt((data_clean['residual'] ** 2).mean())

print(f"📊 Model Performance:")
print(f"   R² (variance explained): {r_squared:.4f} ({r_squared*100:.1f}%)")
print(f"   MAE:  {mae:.3f} tCO₂e/t")
print(f"   RMSE: {rmse:.3f} tCO₂e/t")

print(f"\n✅ Baseline model explains {r_squared*100:.1f}% of emission variance using technology alone")

📊 Model Performance:
   R² (variance explained): 0.7941 (79.4%)
   MAE:  0.220 tCO₂e/t
   RMSE: 0.270 tCO₂e/t

✅ Baseline model explains 79.4% of emission variance using technology alone


### 4.4. Model Building with sklearn
Same analysis using sklearn's built-in functions.

In [7]:
# Encode technology as numbers (0 = EAF, 1 = BF-BOF)
data_clean['tech_encoded'] = (data_clean['tech_binary'] == 'BF-BOF').astype(int)

# Create and fit the model
X = data_clean[['tech_encoded']]
y = data_clean['intensity']

model = DecisionTreeRegressor(max_depth=1, random_state=42)  # max_depth=1 = one split
model.fit(X, y)

# Make predictions
data_clean['predicted'] = model.predict(X)

# Calculate metrics using sklearn
r_squared = r2_score(y, data_clean['predicted'])
mae = mean_absolute_error(y, data_clean['predicted'])
rmse = mean_squared_error(y, data_clean['predicted'], squared=False)

print(f"R² = {r_squared:.4f}")
print(f"MAE = {mae:.3f}")
print(f"RMSE = {rmse:.3f}")

print(f"\nModel object: {model}")
print(f"Feature importance: {model.feature_importances_}")

R² = 0.7941
MAE = 0.220
RMSE = 0.270

Model object: DecisionTreeRegressor(max_depth=1, random_state=42)
Feature importance: [1.]


## 5. Summary for Presentation

In [8]:
print("="*80)
print("BASELINE MODEL SUMMARY")
print("="*80)

summary = f"""
MODEL SPECIFICATION:
  • Decision tree with binary technology classification (EAF vs. BF-BOF)
  • Per-company consistent Scope 2 methodology

SAMPLE:
  • {data_clean['company'].nunique()} companies, {len(data_clean)} observations
  • Years: {data_clean['year'].min():.0f}-{data_clean['year'].max():.0f}
  • EAF: {len(eaf_data)} observations | BF-BOF: {len(bof_data)} observations

PREDICTIONS:
  • IF Technology = EAF    → Predict {eaf_mean:.2f} tCO₂e/t
  • IF Technology = BF-BOF → Predict {bof_mean:.2f} tCO₂e/t
  • Difference: {difference:.2f} tCO₂e/t ({pct_lower:.0f}% lower for EAF)

STATISTICAL VALIDATION:
  • T-test: t = {t_stat:.2f}, p < 0.001*** (highly significant)
  • R² = {r_squared:.3f} ({r_squared*100:.1f}% variance explained)
  • MAE = {mae:.3f} tCO₂e/t | RMSE = {rmse:.3f} tCO₂e/t

KEY TAKEAWAY:
  Technology choice alone explains {r_squared*100:.0f}% of emission variance.
  This establishes the ceiling for more complex models incorporating
  external drivers (carbon pricing, policy) and internal factors (commitments).

FOR SLIDE:
  "Baseline model: {r_squared*100:.0f}% of emissions explained by technology (p<0.001***)"
"""

print(summary)

BASELINE MODEL SUMMARY

MODEL SPECIFICATION:
  • Decision tree with binary technology classification (EAF vs. BF-BOF)
  • Per-company consistent Scope 2 methodology

SAMPLE:
  • 11 companies, 84 observations
  • Years: 2013-2024
  • EAF: 22 observations | BF-BOF: 62 observations

PREDICTIONS:
  • IF Technology = EAF    → Predict 0.60 tCO₂e/t
  • IF Technology = BF-BOF → Predict 1.80 tCO₂e/t
  • Difference: 1.21 tCO₂e/t (67% lower for EAF)

STATISTICAL VALIDATION:
  • T-test: t = -17.78, p < 0.001*** (highly significant)
  • R² = 0.794 (79.4% variance explained)
  • MAE = 0.220 tCO₂e/t | RMSE = 0.270 tCO₂e/t

KEY TAKEAWAY:
  Technology choice alone explains 79% of emission variance.
  This establishes the ceiling for more complex models incorporating
  external drivers (carbon pricing, policy) and internal factors (commitments).

FOR SLIDE:
  "Baseline model: 79% of emissions explained by technology (p<0.001***)"



## 6. Export Clean Dataset

Save the prepared dataset for use in subsequent analyses.

In [9]:
# Select key columns for export
export_cols = ['company', 'country', 'technology', 'tech_binary', 'year', 
               'production', 'intensity', 'method_used', 'predicted', 'residual']

data_export = data_clean[export_cols].copy()

# Save to CSV
data_export.to_csv('data/baseline_model_dataset.csv', index=False)

print(f"✅ Clean dataset exported: {len(data_export)} rows, {len(export_cols)} columns")
print(f"   Saved to: data/baseline_model_dataset.csv")

✅ Clean dataset exported: 84 rows, 10 columns
   Saved to: data/baseline_model_dataset.csv
